# Location Protocol Guided Tutorial

Welcome to this guided tutorial on the **astral-sdk** and **Location Protocol Framework**.

In this notebook, we will walk through the end-to-end process of creating cryptographically verifiable location attestations. We'll cover:

1.  **Setup**: Configuring your environment.
2.  **Generating Geospatial Artifacts**: Creating data from APIs (USGS Earthquakes) and interactive maps.
3.  **Extending Payloads**: Adding metadata and linking to IPFS.
4.  **Creating Attestations**: Signing off-chain and on-chain records.
5.  **Verification**: Verifying and querying attestations. _[NOTE: At the moment, the SDK has no built-in verification functionality of attestations.]_
6.  **Example Application**: Putting it all together.

## What is the Location Protocol?

The Location Protocol is an open standard for creating portable, cryptographically signed records that represent spatial information. These attestations can be used for supply chain tracking, environmental monitoring, digital identity, and more.

For more details, visit the [Location Protocol Specification](https://spec.decentralizedgeo.org/).

## 1. Introduction and Setup

First, let's ensure our Deno environment is ready and import the necessary libraries. We'll be using the `astral-sdk` for location attestations and `ethers` for wallet management.

In [1]:
// Import libraries via esm.sh
import { AstralSDK } from "https://esm.sh/@decentralized-geo/astral-sdk";
import { ethers } from "https://esm.sh/ethers@6";
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";

console.log("✅ Libraries imported successfully!");
console.log("Deno version:", Deno.version.deno);


module "C:\utf-8-validate@6.0.5\denonext\package.json" not found
module "C:\bufferutil@4.0.9\denonext\package.json" not found


✅ Libraries imported successfully!
Deno version: 2.5.6


### Interacting with the Ethereum Blockchain

When working with the Ethereum blockchain, an application needs to interact with it through an **RPC provider** such as Infura, Alchemy, or QuickNode, which provides access to Ethereum nodes on your behalf. These providers act as intermediaries, allowing your application to make requests to the Ethereum blockchain and returning responses.

To communicate with these providers, you use an **RPC URL**, which is essentially the address of the Ethereum node you're connecting to. This URL determines which network (like Ethereum mainnet or testnet like Sepolia) your application will interact with.

To ensure secure and authorized access, you’ll need an **RPC API key**, which acts like a password. This key is included in your RPC URL to authenticate your requests and ensure only authorized users can access the network through the provider.

Finally, you’ll choose a network that best [meets your applications specific requirements](https://www.wilsoncenter.org/article/understanding-ethereums-layer-1-and-layer-2-differences-adoption-and-drawbacks) such as security, cost, scalability, and ecosystem compatibility. Each network has it's own **RPC URL** and **RPC API key**, provided by the **RPC provider**.

Generally speaking, [network choices](https://ethereum.org/layer-2/learn/) are between layer 1 (L1) and layer 2 (L2) networks. L1 networks are the most secure and reliable (e.g. Ethereum mainnet), but the gas fees, the cost of a transaction, are high while only processing 15-20 transactions per second. L2 networks (e.g. Optimism, Arbitrum, Base) give up a little bit of security and reliability, but the gas fees are at a fraction of L1 network costs and scale higher for faster processing of transactions, making them ideal for applications that require high throughput and low costs. 

### Wallet Address

A wallet address is a public key that is used to identify a user on the blockchain. Each wallet address has a corresponding private key that is used to sign transactions and prove ownership of the wallet address.

> **Note**: A private key is a secret key that is used to sign transactions and prove ownership of a wallet address.

### Environment Configuration

To create attestations with the astral-sdk, you'll need a wallet private key and an RPC provider API key (e.g., from [Infura](https://infura.io/), [Alchemy](https://alchemy.com/), or [QuickNode](https://www.quicknode.com/)).

#### Why do I need an RPC provider API key?

Any application that interacts with the Ethereum blockchain needs an RPC provider API key. This enables developers to 
This is because the astral-sdk uses the blockchain to store and verify attestations. The astral-sdk will use the RPC provider API key to make requests to the blockchain.

> **Note**: Copy the `.env-example` file and rename it to `.env` in the same directory as this notebook. Fill in the values for the following content:
>
> ```env
> PRIVATE_KEY=your_private_key_here
> RPC_PROVIDER_API_KEY=your_rpc_provider_api_key_here
> RPC_PROVIDER_URL=your_rpc_provider_url_here
> RPC_CHAIN_ID=your_rpc_chain_id_here
> ```

We will load these values now.

In [ ]:
// Load environment variables
const env = await load();
const PRIVATE_KEY = env["PRIVATE_KEY"] || Deno.env.get("PRIVATE_KEY");
const RPC_PROVIDER_API_KEY = env["RPC_PROVIDER_API_KEY"] || Deno.env.get("RPC_PROVIDER_API_KEY");
const RPC_PROVIDER_URL = env["RPC_PROVIDER_URL"] || Deno.env.get("RPC_PROVIDER_URL");

if (!PRIVATE_KEY || !RPC_PROVIDER_API_KEY || !RPC_PROVIDER_URL) {
  console.warn("⚠️ Missing PRIVATE_KEY or RPC_PROVIDER_API_KEY or RPC_PROVIDER_URL. Some examples may not work.");
} else {  
  console.log("✅ Environment variables loaded.");
}

// concatenate rpc provider url and api key
const rpcProviderUrl = RPC_PROVIDER_URL + RPC_PROVIDER_API_KEY;
const RPC_CHAIN_ID = env["RPC_CHAIN_ID"] || Deno.env.get("RPC_CHAIN_ID");

// Grab the name of the chain from the dictionary mapping chainId to chainName
const chainIdToChainName = {
  "8453": "base",
  "10": "optimism",
  "42161": "arbitrum",
  "42220": "celo",
  "11155111": "sepolia",
};
const CHAIN_NAME = chainIdToChainName[RPC_CHAIN_ID];


✅ Environment variables loaded.


## 2. Generating Geospatial Artifacts

Before we can attest to a location, we need the location data itself. This "geospatial artifact" can come from sources such as:

1.  **API Data Feeds**: Automated sensors, government databases, or existing APIs.
2.  **User Input**: Interactive maps where users draw points, lines, or polygons.

Let's explore both.

### Option A: Fetching Data from an API (USGS Earthquakes)

We'll fetch real-time earthquake data, one of the many available [real-time API data feeds](https://www.usgs.gov/products/web-tools/apis) provided by the U.S. Geological Survey (USGS). This simulates a scenario where an automated system may attest to environmental events coming from an authortative source.

In [3]:
// Fetch earthquakes with magnitude 4.5+ from the last month
const USGS_URL = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_month.geojson";

console.log("Fetching earthquake data...");
const response = await fetch(USGS_URL);
const data = await response.json();

// Get the most recent earthquake
const recentQuake = data.features[0];

console.log(`Found ${data.features.length} earthquakes.`);
console.log("Most recent:", recentQuake.properties.title);
console.log("Full geojson feature:\n\n", recentQuake);

// We will use this 'recentQuake' object as our artifact for the attestation later.


Fetching earthquake data...
Found 500 earthquakes.
Most recent: M 4.9 - 5 km SSW of Palca, Peru
Full geojson feature:

 {
  type: "Feature",
  properties: {
    mag: 4.9,
    place: "5 km SSW of Palca, Peru",
    time: 1765149253429,
    updated: 1765150127040,
    tz: null,
    url: "https://earthquake.usgs.gov/earthquakes/eventpage/us6000rt8z",
    detail: "https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/us6000rt8z.geojson",
    felt: null,
    cdi: null,
    mmi: null,
    alert: null,
    status: "reviewed",
    tsunami: 0,
    sig: 369,
    net: "us",
    code: "6000rt8z",
    ids: ",us6000rt8z,",
    sources: ",us,",
    types: ",origin,phase-data,",
    nst: 37,
    dmin: 0.534,
    rms: 0.86,
    gap: 114,
    magType: "mb",
    type: "earthquake",
    title: "M 4.9 - 5 km SSW of Palca, Peru"
  },
  geometry: { type: "Point", coordinates: [ -69.9851, -17.8229, 109.952 ] },
  id: "us6000rt8z"
}


### Option B: Interactive Map Input

In a user-facing application, you might want users to select a location manually. Below is an example of how you could embed a Leaflet map to capture user input.

*Note: In this notebook, the map is primarily for visualization. In a full application, you would capture the **GeoJSON data** from the `draw:created` event and pass that data to the astral-sdk to construct the location payload.*

In [14]:
const mapHtml = `
<!DOCTYPE html>
<html>
<head>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet.draw/1.0.4/leaflet.draw.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet.draw/1.0.4/leaflet.draw.js"></script>
    <style>#map { height: 400px; }</style>
</head>
<body>
    <div id="map"></div>
    <script>
        var map = L.map('map').setView([37.7749, -122.4194], 13);
        L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
            attribution: '&copy; OpenStreetMap contributors'
        }).addTo(map);

        var drawnItems = new L.FeatureGroup();
        map.addLayer(drawnItems);

        var drawControl = new L.Control.Draw({
            edit: {
                featureGroup: drawnItems
            }
        });
        map.addControl(drawControl);

        map.on(L.Draw.Event.CREATED, function (e) {
            var layer = e.layer;
            drawnItems.addLayer(layer);
            console.log("GeoJSON created:", JSON.stringify(layer.toGeoJSON()));
            // In a real app, you would send this GeoJSON to your backend or SDK
        });
    </script>
</body>
</html>
`;

Deno.jupyter.html`${mapHtml}`;


<!DOCTYPE html>

## 3. Extend Location Payload

Now that we have our geospatial artifact (let's use the earthquake data), we need to transform it into a **Location Payload** compatible with the protocol.

We can also attach metadata, such as IPFS Content Identifiers (CIDs) for media or large datasets.

> **Note**: astral-sdk does *not* handle IPFS uploads directly. You should upload your content to IPFS (using services like Storacha or Filebase) separately and obtain the CID.

In [6]:
// Construct the Location Payload from the USGS data
const locationPayload = {
  srs: "EPSG:4326", // Standard WGS84 coordinates
  locationType: "geojson",
  location: {
    type: "Point",
    coordinates: recentQuake.geometry.coordinates.slice(0, 2) // [lon, lat]
  },
  metadata: {
    name: recentQuake.properties.title,
    timestamp: new Date(recentQuake.properties.time).toISOString(),
    magnitude: recentQuake.properties.mag.toString(),
    // Example of linking an IPFS CID (mocked here)
    media: [
      {
        cid: "QmXoypizjW3WknFiJnKLwHCnL72vedxjQkDDP1mXWo6uco",
        mimeType: "application/json",
        description: "Raw USGS Event Data"
      }
    ]
  }
};

console.log("Constructed Payload:", locationPayload);


Constructed Payload: {
  srs: "EPSG:4326",
  locationType: "geojson",
  location: { type: "Point", coordinates: [ -69.9851, -17.8229 ] },
  metadata: {
    name: "M 4.9 - 5 km SSW of Palca, Peru",
    timestamp: "2025-12-07T23:14:13.429Z",
    magnitude: "4.9",
    media: [
      {
        cid: "QmXoypizjW3WknFiJnKLwHCnL72vedxjQkDDP1mXWo6uco",
        mimeType: "application/json",
        description: "Raw USGS Event Data"
      }
    ]
  }
}


## 4. Creating and Submitting Location Attestation

We can now create an attestation. There are two types:

1.  **Off-chain**: A cryptographically signed message. Free, fast, and private. Good for P2P sharing.
2.  **On-chain**: Stored on the Ethereum blockchain (or L2s). Permanent, public, and composable with smart contracts.

Ensure your payload matches the [Location Protocol Specification](https://spec.decentralizedgeo.org/specification/).

In [7]:
// Initialize SDK with a random wallet if no env vars are set (for demo purposes)
let wallet;
if (PRIVATE_KEY) {
    wallet = new ethers.Wallet(PRIVATE_KEY);
} else {
    console.log("Creating temporary random wallet for demo...");
    wallet = ethers.Wallet.createRandom();
}

const sdk = new AstralSDK({ signer: wallet });

// 1. Create Off-chain Attestation
console.log("Creating off-chain attestation...");
const offchainRecord = await sdk.createOffchainLocationAttestation({
    ...locationPayload,
    memo: "USGS Earthquake Report (Off-chain)"
});

console.log("✅ Off-chain Record Created!");
console.log("UID:", offchainRecord.uid);
console.log("Signature:", offchainRecord.signature);


Creating off-chain attestation...
✅ Off-chain Record Created!
UID: 0x23174f73c70c31779f6c45cfd4feae522eee3758ae6493d6e74692d516f39cb5
Signature: {"v":27,"r":"0x2f1552634861d57fb0290ef7e7b89502d08767e25ff95387715fb0cc0429c8d3","s":"0x6fbc203b400a701c38cffec2abc7b94424294ae570787c30294cf6c054a2e590"}


In [ ]:
// 2. Create On-chain Attestation (Requires valid config)
if (PRIVATE_KEY && RPC_PROVIDER_API_KEY) {
    console.log("\nCreating on-chain attestation (Network: ${CHAIN_NAME})...");

    const provider = new ethers.JsonRpcProvider(`${rpcProviderUrl}`);
    const connectedWallet = wallet.connect(provider);

    const onchainSdk = new AstralSDK({
        signer: connectedWallet,
        chainId: CHAIN_ID
    });

    try {
        const onchainRecord = await onchainSdk.createOnchainLocationAttestation({
            ...locationPayload,
            memo: "USGS Earthquake Report (On-chain)"
        });

        console.log("✅ On-chain Record Created!");
        console.log("Tx Hash:", onchainRecord.txHash);
        console.log(`View on EAS Scan: https://${CHAIN_NAME}.easscan.org/attestation/view/${onchainRecord.uid}`);
    } catch (e) {
        console.error("Error creating on-chain attestation:", e.message);
    }
} else {
    console.log("\n⚠️ Skipping on-chain creation (missing configuration).");
}



Creating on-chain attestation (Sepolia)...
✅ On-chain Record Created!
Tx Hash: 0x9a32d0ce7b8e4a36cb6086c694a94584014a999044235ba972cb105e31fe11c1
View on EAS Scan: https://sepolia.easscan.org/attestation/view/0x42d91d97f45018d3fc804d5b817d743685cfb045b2d9067ab33fc96e362e3ba6


## 5. Attestation Verification and Query

> As previously mentioned, the SDK does not include built-in verification functionality for attestations.

Verification is crucial. It ensures the data hasn't been tampered with and was signed by the expected entity.

We can inspect the response object and verify the signature.

> NOTE: I may need to create some custom methods to showcase verification. The approach below is a pattern that I've seen with the etherscan verified signatures tool: **https://info.etherscan.com/verify-signature-tool/**

In [ ]:
// Verify the off-chain record we just created
console.log("Verifying record...");

// In a real scenario, you would receive 'offchainRecord' from an external source
const isValid = await sdk.validateLocationAttestation(offchainRecord);

console.log(`Is the record valid? ${isValid ? "✅ YES" : "❌ NO"}`);

// You can also recover the signer's address to ensure it came from a trusted source
const recoveredAddress = ethers.verifyMessage(
    offchainRecord.messageHash || "mock_hash", // SDK usually handles hashing internally during verification
    offchainRecord.signature
);
// Note: The SDK's validate method handles the complex EIP-712 reconstruction and verification for you.


## 6. Example Applications

You can build powerful applications by combining these primitives. For a more complex, full-stack example, check out the [Privy + EAS Integration Demo](https://github.com/DecentralizedGeo/privy-eas-integration-demo).

Below is a simplified flow of how an app might handle a user checking in:

In [11]:
async function handleUserCheckin(userWallet, coordinates, note) {
    console.log(`Processing check-in for ${userWallet.address}...`);
    
    const appSdk = new AstralSDK({ signer: userWallet });
    
    const checkin = await appSdk.createOffchainLocationAttestation({
        location: { 
            type: 'Point', 
            coordinates: coordinates 
        },
        memo: note
    });
    
    return checkin.uid;
}

// Simulate a check-in
const checkinId = await handleUserCheckin(wallet, [-73.935242, 40.730610], "Coffee shop visit");
console.log("New Check-in ID:", checkinId);


Processing check-in for 0x3074C8732366cE5DB80986aBA8FB69897872DdB9...
New Check-in ID: 0x2c6e25f05a4a4833697ce486b2e0579827c81f89b8960f3aea6a0039234450b7


## Wrapping Up

You've now learned how to:
1.  Fetch geospatial data from APIs.
2.  Capture user location input.
3.  Create verifiable location attestations (on and off-chain).
4.  Verify these records.

The astral-sdk makes it easy to integrate verifiable location data into your dApps, supply chain systems, and more.